CASE STUDY 2: CREDIT CARD FRAUD DETECTION

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, average_precision_score
)


In [4]:
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

In [ ]:
data=pd.read_csv("C:\Jaykesh\Coding\ML_Assignment\creditcard.csv")
print(data)

In [ ]:
print("Dataset Shape:", data.shape)
print("\nFrist 5 rows:")
print(data.head())
print("\nMissing Values:")
print(data.isnull().sum().sum())
print("\nClass Distributation:")
print(data["Class"].mean()*100)


In [8]:
X=data.drop("Class", axis=1)
y=data["Class"]



In [9]:
scaler=StandardScaler()
X["Amount"]=scaler.fit_transform(X[["Amount"]])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)

In [ ]:
print("\nBefore SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

In [ ]:
fraud_count = sum(y_train_smote == 1)
normal_count = sum(y_train_smote == 0)

scale_pos_weight = normal_count / fraud_count

print("\nScale Positive Weight:", scale_pos_weight)

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42
)

model.fit(
    X_train_smote,
    y_train_smote
)

print("\nXGBoost model training completed!")

In [ ]:
y_probability = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_probability)

print("\nROC-AUC Score:", roc_auc)

pr_auc = average_precision_score(y_test, y_probability)

print("PR-AUC Score:", pr_auc)

In [ ]:
threshold = 0.50

y_pred_default = (y_probability >= threshold).astype(int)

print("\nClassification Report - Threshold 0.50")
print(classification_report(y_test, y_pred_default))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_default))

precision, recall, thresholds = precision_recall_curve(
    y_test,
    y_probability
)

In [ ]:
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)

best_index = np.argmax(f1_scores)
best_threshold = thresholds[best_index]

print("\nBest Threshold:", best_threshold)
print("Best F1 Score:", f1_scores[best_index])

y_pred_tuned = (
    y_probability >= best_threshold
).astype(int)

print("\nClassification Report - Tuned Threshold")
print(classification_report(y_test, y_pred_tuned))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

In [ ]:
print("\nThreshold Comparison:")

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:

    prediction = (y_probability >= threshold).astype(int)

    report = classification_report(
        y_test,
        prediction,
        output_dict=True,
        zero_division=0
    )

    fraud_precision = report["1"]["precision"]
    fraud_recall = report["1"]["recall"]
    fraud_f1 = report["1"]["f1-score"]

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {fraud_precision:.3f} | "
        f"Recall: {fraud_recall:.3f} | "
        f"F1: {fraud_f1:.3f}"
    )

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(
    recall,
    precision
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid()

plt.show()

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
)

importance = importance.sort_values(
    ascending=False
)

print("\nTop 15 Important Features:")
print(importance.head(15))

plt.figure(figsize=(10, 6))

importance.head(15).sort_values().plot(
    kind="barh"
)

plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.title("Top 15 Feature Importance - XGBoost")

plt.show()

In [ ]:
print("\n")
print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)
print("Best Threshold:", best_threshold)
print("Best F1 Score:", f1_scores[best_index])
